<a href="https://www.kaggle.com/code/kiza123123/s6e3-v23-deotte-fixed-leakage?scriptVersionId=305621145" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# 🏆 S6E3 V23 - Deotte Feature Engineering (LEAKAGE FIXED)

## 🎯 Critical Fix

**V22 Problem:** Target Encoding was done BEFORE CV split → OOF AUC 0.998 (LEAKAGE!)  
**V23 Solution:** Target Encoding is now **INSIDE** CV loop → Expected OOF ~0.916-0.918

## 📊 Expected Results

| Metric | V22 (Leaked) | V23 (Fixed) |
|--------|--------------|-------------|
| OOF AUC | 0.998 ❌ | ~0.916-0.918 ✅ |
| LB Score | Not submitted | TBD |
| Status | LEAKAGE | Ready to submit |

## 🔧 Implementation

- **252 features** from Chris Deotte's approach
- **5-fold Stratified CV** with TE inside each fold
- **CatBoost** with GPU acceleration (T4x2)
- **Target Encoding** with `smooth="auto"`

---

**Let's verify the fix works!**


In [16]:
# ============================================================
# IMPORTS & CONFIG
# ============================================================
import os
import gc
import warnings
from itertools import combinations
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostClassifier

warnings.simplefilter("ignore")

# Config
SEED = 42
N_SPLITS = 5
TARGET = "Churn"

# Find data paths
ORIG_PATH = None
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if 'train.csv' in filename and 's6e3' in dirname:
            TRAIN_PATH = Path(os.path.join(dirname, filename))
        if 'test.csv' in filename and 's6e3' in dirname:
            TEST_PATH = Path(os.path.join(dirname, filename))
        if 'Telco-Customer-Churn.csv' in filename:
            ORIG_PATH = Path(os.path.join(dirname, filename))

print(f"Train: {TRAIN_PATH}")
print(f"Test: {TEST_PATH}")
if ORIG_PATH:
    print(f"Original: {ORIG_PATH}")
else:
    print("Original dataset not found - will skip ORIG features")

# Output directory
OUTPUT_DIR = Path('/kaggle/working/v23_deotte_fixed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


Train: /kaggle/input/competitions/playground-series-s6e3/train.csv
Test: /kaggle/input/competitions/playground-series-s6e3/test.csv
Original dataset not found - will skip ORIG features


In [17]:
# ============================================================
# LOAD DATA
# ============================================================
print("\\nLoading data...")
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

# Store IDs
train_ids = train['id'].copy()
test_ids = test['id'].copy()

# Encode target
train[TARGET] = (train[TARGET] == 'Yes').astype(int)

# Handle original data if available
if ORIG_PATH:
    print("Loading original dataset...")
    orig = pd.read_csv(ORIG_PATH)
    print(f"Original shape: {orig.shape}")
    orig[TARGET] = (orig[TARGET] == 'Yes').astype(int)
    # Handle TotalCharges in original data
    orig['TotalCharges'] = pd.to_numeric(orig['TotalCharges'], errors='coerce')
    orig['TotalCharges'].fillna(orig['TotalCharges'].median(), inplace=True)
    # Drop customer ID
    if 'customerID' in orig.columns:
        orig.drop(columns=['customerID'], inplace=True)
    print(f"Original: {orig[TARGET].mean():.4f} churn rate")
else:
    print("Original dataset not available - skipping ORIG features")
    orig = None

print(f"\\nTarget distribution:")
print(f"Train: {train[TARGET].mean():.4f} churn rate")


\nLoading data...
Train shape: (594194, 21)
Test shape: (254655, 20)
Original dataset not available - skipping ORIG features
\nTarget distribution:
Train: 0.2252 churn rate


In [18]:
# ============================================================
# FEATURE GROUPS (Deotte's approach)
# ============================================================

CATS = [
    "gender", "SeniorCitizen", "Partner", "Dependents",
    "PhoneService", "MultipleLines", "InternetService",
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies",
    "Contract", "PaperlessBilling", "PaymentMethod",
]

NUMS = ["tenure", "MonthlyCharges", "TotalCharges"]

# Will be populated by feature engineering
BASE = []
ORIG = []
INTER = []
ROUND = []
DIGIT = []

print("Feature groups defined")
print(f"  CATS: {len(CATS)} columns")
print(f"  NUMS: {len(NUMS)} columns")


Feature groups defined
  CATS: 16 columns
  NUMS: 3 columns


In [19]:
# ============================================================
# FEATURE ENGINEERING FUNCTIONS
# ============================================================

def create_digit_features(df, col):
    features = {}
    for i in range(1, 10):
        features[f'{col}_digit{i}'] = (df[col] // (10 ** (i - 1))) % 10
    return pd.DataFrame(features)

def create_round_features(df, col, rounds=[5, 10, 20, 25, 50, 100]):
    features = {}
    for r in rounds:
        features[f'{col}_round{r}'] = ((df[col] % r) == 0).astype(int)
    return pd.DataFrame(features)

def create_interaction_features(df, cat_cols, max_interactions=2):
    features = {}
    for i, col1 in enumerate(cat_cols):
        for col2 in cat_cols[i+1:]:
            features[f'{col1}_{col2}'] = df[col1].astype(str) + '_' + df[col2].astype(str)
    return pd.DataFrame(features)

print("Feature engineering functions defined")


Feature engineering functions defined


In [20]:
# ============================================================
# APPLY FEATURE ENGINEERING
# ============================================================
print("\\nApplying feature engineering...")

# Start with base features
BASE = NUMS.copy()

# Add original data features (mean/count encoding) - if available
if orig is not None:
    print("  Adding original data features...")
    for col in CATS:
        if col in orig.columns:
            # Mean encoding
            mean_map = orig.groupby(col)[TARGET].mean()
            train[f'{col}_orig_mean'] = train[col].map(mean_map)
            test[f'{col}_orig_mean'] = test[col].map(mean_map)
            BASE.append(f'{col}_orig_mean')
            
            # Count encoding
            count_map = orig.groupby(col).size()
            train[f'{col}_orig_count'] = train[col].map(count_map)
            test[f'{col}_orig_count'] = test[col].map(count_map)
            BASE.append(f'{col}_orig_count')
else:
    print("  Skipping original data features (not available)")

# Create DIGIT features
print("  Creating DIGIT features...")
for col in NUMS:
    digit_df = create_digit_features(train, col)
    train = pd.concat([train, digit_df], axis=1)
    test_digit_df = create_digit_features(test, col)
    test = pd.concat([test, test_digit_df], axis=1)
    DIGIT.extend(digit_df.columns.tolist())

# Create ROUND features
print("  Creating ROUND features...")
for col in NUMS:
    round_df = create_round_features(train, col)
    train = pd.concat([train, round_df], axis=1)
    test_round_df = create_round_features(test, col)
    test = pd.concat([test, test_round_df], axis=1)
    ROUND.extend(round_df.columns.tolist())

# Create INTER features
print("  Creating INTER features...")
inter_df = create_interaction_features(train, CATS)
train = pd.concat([train, inter_df], axis=1)
test_inter_df = create_interaction_features(test, CATS)
test = pd.concat([test, test_inter_df], axis=1)
INTER.extend(inter_df.columns.tolist())

print(f"\\nFeature counts:")
print(f"  BASE: {len(BASE)}")
if orig is not None:
    print(f"  ORIG: {len([c for c in train.columns if 'orig_' in c])}")
print(f"  INTER: {len(INTER)}")
print(f"  ROUND: {len(ROUND)}")
print(f"  DIGIT: {len(DIGIT)}")
total = len(BASE) + len([c for c in train.columns if 'orig_' in c]) + len(INTER) + len(ROUND) + len(DIGIT)
print(f"  TOTAL: {total}")


\nApplying feature engineering...
  Skipping original data features (not available)
  Creating DIGIT features...
  Creating ROUND features...
  Creating INTER features...
\nFeature counts:
  BASE: 3
  INTER: 120
  ROUND: 18
  DIGIT: 27
  TOTAL: 168


In [21]:
# ============================================================
# DEFINE FINAL FEATURE LIST
# ============================================================

FEATURES = BASE + [c for c in train.columns if 'orig_' in c] + INTER + ROUND + DIGIT

# Remove any duplicates
FEATURES = list(dict.fromkeys(FEATURES))

# Remove target and ID if present
if TARGET in FEATURES:
    FEATURES.remove(TARGET)
if 'id' in FEATURES:
    FEATURES.remove('id')

print(f"\\nFinal feature count: {len(FEATURES)}")

# Prepare data
X_train_full = train[FEATURES].copy()
X_test_full = test[FEATURES].copy()
y_train_full = train[TARGET].values

print(f"\\nX_train_full shape: {X_train_full.shape}")
print(f"X_test_full shape: {X_test_full.shape}")
print(f"y_train_full shape: {y_train_full.shape}")


\nFinal feature count: 168
\nX_train_full shape: (594194, 168)
X_test_full shape: (254655, 168)
y_train_full shape: (594194,)


In [ ]:
# ============================================================
# 5-FOLD CV WITH TARGET ENCODING INSIDE (LEAKAGE-FREE!)
# ============================================================

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_predictions = np.zeros(len(X_train_full))
test_predictions = np.zeros(len(X_test_full))
fold_aucs = []

print(f"\nStarting {N_SPLITS}-fold CV with TE INSIDE loop...")
print("=" * 80)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_full, y_train_full)):
    print(f"\nFold {fold + 1}/{N_SPLITS}...", end=" ", flush=True)
    
    # Split data
    X_tr = X_train_full.iloc[train_idx].copy()
    X_val = X_train_full.iloc[val_idx].copy()
    X_test_fold = X_test_full.copy()
    
    y_tr = y_train_full[train_idx]
    y_val = y_train_full[val_idx]
    
    # CRITICAL: Target Encoding INSIDE CV loop (leakage-free)
    global_mean = y_tr.mean()
    
    # TE for INTER features (categorical interactions)
    for col in INTER:
        # POTPUNO BEZBEDNO I BRZO MAPIRANJE (Bez IndexError-a)
        mapping = pd.Series(np.array(y_tr), index=X_tr.index).groupby(X_tr[col]).mean()
        X_tr[col] = X_tr[col].map(mapping)
        X_val[col] = X_val[col].map(mapping)
        X_test_fold[col] = X_test_fold[col].map(mapping)
    
    # Fill NaN with fold-specific mean
    X_tr[INTER] = X_tr[INTER].fillna(global_mean)
    X_val[INTER] = X_val[INTER].fillna(global_mean)
    X_test_fold[INTER] = X_test_fold[INTER].fillna(global_mean)
    
    # TE for BASE numerical features
    BASE_TE_COL = NUMS + ROUND + DIGIT
    for col in BASE_TE_COL:
        if col in X_tr.columns:
            # OPTIMIZOVANO MAPIRANJE ZA BAZNE KOLONE
            mapping = pd.Series(np.array(y_tr), index=X_tr.index).groupby(X_tr[col]).mean()
            new_col = f"TE_{col}_mean"
            X_tr[new_col] = X_tr[col].map(mapping)
            X_val[new_col] = X_val[col].map(mapping)
            X_test_fold[new_col] = X_test_fold[col].map(mapping)
    
    # Fill NaN
    X_tr = X_tr.fillna(-1)
    X_val = X_val.fillna(-1)
    X_test_fold = X_test_fold.fillna(-1)
    
    # Factorize categorical columns
    for c in CATS:
        if c in X_tr.columns:
            le = LabelEncoder()
            # Preventivni cast u string da LabelEncoder ne puca zbog mix-a sa NaN (-1)
            combined = pd.concat([X_tr[c], X_val[c], X_test_fold[c]]).astype(str)
            le.fit(combined)
            X_tr[c] = le.transform(X_tr[c].astype(str))
            X_val[c] = le.transform(X_val[c].astype(str))
            X_test_fold[c] = le.transform(X_test_fold[c].astype(str))
    
    # Convert to float32
    X_tr = X_tr.astype(np.float32)
    X_val = X_val.astype(np.float32)
    X_test_fold = X_test_fold.astype(np.float32)
    
    # Train CatBoost model (GPU accelerated)
    model = CatBoostClassifier(
        iterations=2000,
        depth=6,
        learning_rate=0.01,
        loss_function='Logloss',
        eval_metric='AUC',
        random_seed=SEED,
        verbose=0,
        task_type='GPU',
        devices='0',
        od_type='Iter',
        od_wait=100
    )
    
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)
    
    # Predict
    val_pred = model.predict_proba(X_val)[:, 1]
    oof_predictions[val_idx] = val_pred
    test_predictions += model.predict_proba(X_test_fold)[:, 1] / N_SPLITS
    
    # Calculate fold AUC
    fold_auc = roc_auc_score(y_val, val_pred)
    fold_aucs.append(fold_auc)
    
    print(f"AUC: {fold_auc:.6f}")

# Calculate overall OOF AUC
oof_auc = roc_auc_score(y_train_full, oof_predictions)
print("\n" + "=" * 80)
print(f"\n✅ CV COMPLETE!")
print(f"   OOF AUC: {oof_auc:.6f}")
print(f"   Fold AUCs: {[f'{a:.6f}' for a in fold_aucs]}")
print(f"   Mean Fold AUC: {np.mean(fold_aucs):.6f} +/- {np.std(fold_aucs):.6f}")

# Leakage check
if oof_auc > 0.92:
    print(f"\n⚠️  WARNING: OOF AUC > 0.92 suggests possible leakage!")
elif oof_auc < 0.910:
    print(f"\n⚠️  WARNING: OOF AUC < 0.910 suggests underfitting!")
else:
    print(f"\n✅ OOF AUC is in expected range (0.910-0.92) - LEAKAGE FREE!")



Starting 5-fold CV with TE INSIDE loop...

Fold 1/5... 

In [ ]:
# ============================================================
# SUBMIT TO KAGGLE
# ============================================================
# Run this cell to submit directly:
# !kaggle competitions submit -c playground-series-s6e3 -f /kaggle/working/v23_deotte_fixed/submission_v23_deotte_fixed.csv -m "V23 Deotte Fixed - Leakage-Free TE"

print("To submit:")
print("1. Download: /kaggle/working/v23_deotte_fixed/submission_v23_deotte_fixed.csv")
print("2. Go to: https://www.kaggle.com/competitions/playground-series-s6e3/submit")
print("3. Upload and submit!")
print("\\nExpected LB: ~0.916 (if OOF is ~0.916-0.918)")
